In [14]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import dagshub
import seaborn as sns
import pandas as pd
import warnings
from sklearn.model_selection import cross_val_score

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/Sentiment analysis /cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [6]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9a7305a4-df15-4d8a-a231-f45cc2378663&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=59cb6166a779484c727ae844ac227fbb9e96f87d6d56c17abe4b193e0ead007c




Accessing as vaibhav-patel01

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [7]:
mlflow.set_experiment("Traning_LGBM")


<Experiment: artifact_location='mlflow-artifacts:/98341e5c5d954bd1b4456820fe5026d0', creation_time=1783601308550, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783601308550, lifecycle_stage='active', name='Traning_LGBM', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [16]:
warnings.filterwarnings("ignore", category=UserWarning)

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 5000   # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['CommentText'])
y = df['Sentiment']

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [10]:
model = LGBMClassifier(n_estimators=259, learning_rate=0.05927859213070258, n_jobs= 6, max_depth=8 ,class_weight= "balanced", verbose=-1)

In [15]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6117730683164407


In [13]:
classification_rep = classification_report(y_test, y_pred, output_dict=True)
print(classification_rep)

{'0': {'precision': 0.615572795362008, 'recall': 0.5802783086667404, 'f1-score': 0.597404709249382, 'support': 63311.0}, '1': {'precision': 0.5203210280541257, 'recall': 0.6881970113417369, 'f1-score': 0.5925991417126302, 'support': 63306.0}, '2': {'precision': 0.7715262399759207, 'recall': 0.5668477917482783, 'f1-score': 0.6535362089217909, 'support': 63308.0}, 'accuracy': 0.6117730683164407, 'macro avg': {'precision': 0.6358066877973515, 'recall': 0.6117743705855853, 'f1-score': 0.6145133532946011, 'support': 189925.0}, 'weighted avg': {'precision': 0.6358075843072484, 'recall': 0.6117730683164407, 'f1-score': 0.6145133138184448, 'support': 189925.0}}


In [ ]:
# Function to optimize LightGBM hyperparameters
def objective(trial):
    # Define hyperparameters to be tuned
    param = {
        "objective": "multiclass",
        "num_class": 3,  # Assuming 3 categories (-1, 0, 1)
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "metric": "multi_logloss",
        "is_unbalance": True,
        "class_weight": "balanced",
    }

    # Define the LightGBM model with the trial parameters
    model = LGBMClassifier(**param)

    # Perform cross-validation
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')

    # Return the average score across folds
    return scores.mean()


# Create an Optuna study to optimize the hyperparameters
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)



In [ ]:

# Extract the best hyperparameters
best_params = study.best_params
best_params

In [ ]:

best_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.1,   # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    learning_rate=0.08,
    max_depth=20,
    n_estimators=367
)


# Fit the model on the resampled training data
best_model.fit(X_train, y_train)

In [11]:
warnings.filterwarnings("ignore", category=UserWarning)
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params=None, trial_num=None):
    with mlflow.start_run():
        # Log model type and tags
        run_name = f"{model_name}_TFIDF_Trigrams"
        if trial_num is not None:
            run_name += f"_Trial_{trial_num}"

        mlflow.set_tag("mlflow.runName", run_name)
        mlflow.set_tag("experiment_type", "hyperparameters comparison")

        # Log algorithm name and hyperparameters
        mlflow.log_param("algo_name", model_name)
        if params:
            mlflow.log_params(params) # CRITICAL: Actually log the Optuna parameters

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, model={model_name}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

        # CRITICAL: Return the metric so Optuna can evaluate the trial
        return accuracy

In [14]:
warnings.filterwarnings("ignore", category=UserWarning)
# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)

    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda
    }

    model = LGBMClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           max_depth=max_depth,
                           num_leaves=num_leaves,
                           min_child_samples=min_child_samples,
                           colsample_bytree=colsample_bytree,
                           subsample=subsample,
                           reg_alpha=reg_alpha,
                           reg_lambda=reg_lambda, clas random_state=42, verbose=-1)

    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)
    return accuracy

In [15]:
# Step 7: Run Optuna for LightGBM, log the best model, and plot the importance of each parameter
warnings.filterwarnings("ignore", category=UserWarning)
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100) # Increased to 100 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'],
                                learning_rate=best_params['learning_rate'],
                                max_depth=best_params['max_depth'],
                                num_leaves=best_params['num_leaves'],
                                min_child_samples=best_params['min_child_samples'],
                                colsample_bytree=best_params['colsample_bytree'],
                                subsample=best_params['subsample'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'],

                                class_weight='balanced',
                                random_state=42)

    # Log the best model with MLflow and print the classification report
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best")

    # Plot parameter importance
    optuna.visualization.plot_param_importances(study).show()

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study).show()

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-09 13:15:26,261] A new study created in memory with name: no-name-65fc32f4-c313-46e5-afa0-3b5db98de83b
[W 2026-07-09 13:17:34,910] Trial 0 failed with parameters: {'n_estimators': 490, 'learning_rate': 0.017088591962378434, 'max_depth': 10, 'num_leaves': 122, 'min_child_samples': 93, 'colsample_bytree': 0.7992455395400493, 'subsample': 0.6504615084644256, 'reg_alpha': 0.068862937841665, 'reg_lambda': 0.0006044160087482075} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_524/4086020093.py", line 36, in objective_lightgbm
    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_524/180

🏃 View run LightGBM_TFIDF_Trigrams_Trial_0 at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/4/runs/b65586fb280142f88c645f51c12daaff
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/4


KeyboardInterrupt: 

In [2]:
pip install mlflow dagshub optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 135.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11